# Test Unrecognized Faces API

This notebook tests creating and viewing unrecognized face records.

**Features tested:**
- Upload unrecognized face images
- List unrecognized faces with signed URLs
- Verify image accessibility
- Test retention cleanup with old records

**Before running:**
1. Make sure backend is running
2. Have test images ready (optional)
3. Login as ORG ADMIN

## Setup

In [ ]:
import os
import io
import cv2
import numpy as np
import requests
from datetime import datetime, timezone, timedelta
from dotenv import load_dotenv

load_dotenv()

BACKEND_URL = os.getenv('SO_BACKEND_API_URL', 'http://localhost:7091')
session = requests.Session()
session.headers.update({"accept": "application/json"})

print('Setup complete!')
print(f'Backend API: {BACKEND_URL}')

## Login

In [ ]:
def login_to_backend(email=None, password=None, client_slug='humblebee'):
    if email is None:
        email = os.getenv('SO_ADMIN_EMAIL', 'admin@humblebee.ai')
    if password is None:
        password = os.getenv('SO_ADMIN_PASSWORD', 'admin123')

    print(f'Logging in as {email} to org "{client_slug}"...')

    response = session.post(
        f'{BACKEND_URL}/api/auth/login',
        json={'email': email, 'password': password, 'client_slug': client_slug}
    )

    if response.status_code == 200:
        data = response.json()
        token = data.get('token') or data.get('accessToken')
        session.headers.update({'Authorization': f'Bearer {token}'})
        print('Login successful')
        print(f"  User: {data.get('user', {}).get('full_name')}")
        return token, client_slug
    else:
        print(f'Login failed: {response.status_code}')
        return None, None

auth_token, slug = login_to_backend(
    email='humblebee@gmail.com',
    password='Humblebee2025@',
    client_slug='humblebee'
)

## Helper: Create Test Image

In [ ]:
def create_test_image(text="UNRECOGNIZED", size=(400, 400)):
    """Create a test image with text overlay"""
    img = np.zeros((size[1], size[0], 3), dtype=np.uint8)
    img[:, :] = [100, 150, 200]  # Blue background

    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    cv2.putText(img, text, (50, 150), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 3)
    cv2.putText(img, timestamp, (50, 250), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (200, 200, 200), 2)

    _, encoded = cv2.imencode('.jpg', img)
    return encoded.tobytes()

print('Test image helper ready!')

## Test 1: Upload Unrecognized Face (Current Time)

In [ ]:
def upload_unrecognized_face(image_path=None, camera_id=None):
    url = f"{BACKEND_URL}/api/org/{slug}/unrecognized-faces"
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    # Prepare image
    if image_path and os.path.exists(image_path):
        print(f'Using image: {image_path}')
        with open(image_path, 'rb') as f:
            image_data = f.read()
        filename = os.path.basename(image_path)
    else:
        print('Creating test image...')
        image_data = create_test_image("UNRECOGNIZED FACE")
        filename = 'test_unrecognized.jpg'

    files = [('images', (filename, io.BytesIO(image_data), 'image/jpeg'))]
    data = {
        'detection_time': timestamp,
        'user_status': 'in',
        'notes': 'Test unrecognized face'
    }

    if camera_id:
        data['camera_id'] = str(camera_id)

    headers = {"Authorization": session.headers.get("Authorization")}
    print(f'Uploading to: {url}')

    r = requests.post(url, headers=headers, files=files, data=data, timeout=30)
    print(f'Status: {r.status_code}')

    if r.status_code in [200, 201]:
        result = r.json()
        print('Upload successful!')
        print(f'  ID: {result.get("id")}')
        print(f'  Detection Time: {result.get("detection_time")}')
        print(f'  Image URL: {result.get("image_url", "N/A")[:100]}...')
        return result
    else:
        print(f'Upload failed: {r.text[:300]}')
        return None

# Test upload (change path to use your own image)
test_image_path = None  # Set to your image path or leave None for test image
result = upload_unrecognized_face(image_path=test_image_path)

## Test 2: List and Verify Unrecognized Faces

In [ ]:
def list_and_verify_unrecognized_faces(limit=5):
    url = f"{BACKEND_URL}/api/org/{slug}/unrecognized-faces"

    print(f'Fetching unrecognized faces (limit={limit})...')

    r = session.get(url, params={"limit": limit})
    r.raise_for_status()

    data = r.json()
    unrecognized_users = data.get('unrecognized_users', [])
    stats = data.get('stats', {})

    print(f'\nFound {len(unrecognized_users)} records')
    print(f'Stats: {stats}')
    print('\n' + '='*70)

    for i, record in enumerate(unrecognized_users[:limit], 1):
        print(f'\n--- Record {i} ---')
        print(f'ID: {record.get("id")}')
        print(f'Detection Time: {record.get("detection_time")}')
        print(f'Status: {record.get("status")}')
        print(f'User Status: {record.get("user_status", "N/A")}')

        image_url = record.get('image_url', '')
        if image_url:
            is_signed = "X-Goog-Signature" in image_url or "Signature=" in image_url
            has_expiry = "Expires=" in image_url or "X-Goog-Expires" in image_url

            print(f'\nImage URL:')
            print(f'  Preview: {image_url[:100]}...')
            print(f'  Is Signed: {"YES" if is_signed else "NO"}')
            print(f'  Has Expiry: {"YES" if has_expiry else "NO"}')

            if is_signed:
                try:
                    test_r = requests.head(image_url, timeout=5)
                    print(f'  Accessible: {"YES (HTTP {test_r.status_code})" if test_r.status_code == 200 else f"HTTP {test_r.status_code}"}')
                except Exception as e:
                    print(f'  Accessible: ERROR - {str(e)[:50]}')
        else:
            print('No image_url in record')

    print('\n' + '='*70)
    return data

result = list_and_verify_unrecognized_faces(limit=3)

## Test 3: Upload Multiple Unrecognized Faces

In [ ]:
import time

print('Creating multiple test unrecognized faces...')
print('='*70)

test_cases = [
    {'label': 'Person A', 'status': 'in'},
    {'label': 'Person B', 'status': 'out'},
    {'label': 'Person C', 'status': 'in'},
]

created_ids = []
for i, test in enumerate(test_cases, 1):
    print(f'\n{i}. Creating {test["label"]} ({test["status"]})...')

    image_data = create_test_image(test['label'])
    timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    files = [('images', (f'{test["label"]}.jpg', io.BytesIO(image_data), 'image/jpeg'))]
    data = {
        'detection_time': timestamp,
        'user_status': test['status'],
        'notes': f'Test: {test["label"]}'
    }

    url = f"{BACKEND_URL}/api/org/{slug}/unrecognized-faces"
    headers = {"Authorization": session.headers.get("Authorization")}

    r = requests.post(url, headers=headers, files=files, data=data, timeout=30)
    if r.status_code in [200, 201]:
        result = r.json()
        created_ids.append(result.get('id'))
        print(f'  Created - ID: {result.get("id")}')
    else:
        print(f'  Failed: {r.status_code}')

    time.sleep(0.5)

print('\n' + '='*70)
print(f'Created {len(created_ids)} unrecognized face records')
print('Check the Unrecognized Users page in the frontend!')

## Test 4: Create Old Records (For Retention Testing)

In [ ]:
def create_old_unrecognized_face(days_old, label="OLD"):
    """Create an unrecognized face with a specific date in the past"""

    detection_date = datetime.now(timezone.utc) - timedelta(days=days_old)
    detection_time = detection_date.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"

    image_data = create_test_image(f'{label}\n{days_old}d ago')

    url = f"{BACKEND_URL}/api/org/{slug}/unrecognized-faces"
    files = [('images', (f'test_{days_old}d.jpg', io.BytesIO(image_data), 'image/jpeg'))]
    data = {
        'detection_time': detection_time,
        'user_status': 'in',
        'notes': f'Test - {days_old} days old for retention testing'
    }

    headers = {"Authorization": session.headers.get("Authorization")}

    r = requests.post(url, headers=headers, files=files, data=data, timeout=30)
    if r.status_code in [200, 201]:
        result = r.json()
        print(f'Created {days_old}d old record - ID: {result.get("id")}, Date: {detection_time}')
        return result
    else:
        print(f'Failed to create {days_old}d old record: {r.text[:200]}')
        return None

print('Creating test records with various ages...')
print('='*70)

test_ages = [
    (5, "5 DAYS"),
    (30, "30 DAYS"),
    (60, "60 DAYS"),
    (95, "95 DAYS"),
]

created_ids = []
for days, label in test_ages:
    result = create_old_unrecognized_face(days, label)
    if result:
        created_ids.append((result.get('id'), days))

print('\n' + '='*70)
print(f'Created {len(created_ids)} test records:')
for record_id, days in created_ids:
    print(f'  - ID {record_id}: {days} days old')

print('\nYou can now test the retention cleanup feature in the Profile page!')

## Verify Records

In [ ]:
print('Fetching all unrecognized faces...')

r = session.get(f"{BACKEND_URL}/api/org/{slug}/unrecognized-faces", params={'limit': 20})

if r.status_code == 200:
    data = r.json()
    records = data.get('unrecognized_users', [])
    stats = data.get('stats', {})

    print(f'\nFound {len(records)} records')
    print(f'Stats: Total={stats.get("total")}, Pending={stats.get("pending")}')
    print('\n')

    for rec in records[:10]:
        status = rec.get('status', 'pending')
        user_status = rec.get('user_status', 'N/A')
        has_image = 'YES' if rec.get('image_url') else 'NO'

        print(f"{rec['detection_time'][11:19]} | Status: {status:15} | User Status: {user_status:3} | Image: {has_image}")

    print('\nVerification complete!')
else:
    print(f'Failed: {r.status_code}')